# Test Manual del Consorcio - Xcapit FHE-ML Platform

Este notebook prueba el flujo completo del consorcio:
1. Crear usuarios y companies
2. Crear un consorcio
3. Invitar miembros
4. Aceptar invitaciones
5. Registrar contribuciones de datos
6. Iniciar entrenamiento

In [ ]:
import requests
import json
from datetime import datetime, timedelta

# Configuracion
BASE_URL = "http://localhost:8000"
API_URL = f"{BASE_URL}/api/v2"

def print_response(resp, label="Response"):
    """Helper para imprimir respuestas."""
    print(f"\n=== {label} ===")
    print(f"Status: {resp.status_code}")
    try:
        print(json.dumps(resp.json(), indent=2, default=str))
    except:
        print(resp.text[:500])

## 1. Verificar que el servidor esta corriendo

In [ ]:
# Health check
resp = requests.get(f"{BASE_URL}/health/")
print_response(resp, "Health Check")

if resp.status_code != 200:
    print("\n[ERROR] El servidor no esta corriendo!")
    print("Ejecuta: docker compose --profile django-dev up -d")

## 2. Crear usuarios de prueba

Vamos a crear 3 companies con sus usuarios:
- Hospital Alpha (owner del consorcio)
- Hospital Beta (miembro)
- Hospital Gamma (miembro)

In [ ]:
# Datos de prueba para las 3 companies
COMPANIES = [
    {
        "name": "Hospital Alpha",
        "email": "admin@hospital-alpha.test",
        "password": "TestPassword123!"
    },
    {
        "name": "Hospital Beta",
        "email": "admin@hospital-beta.test",
        "password": "TestPassword123!"
    },
    {
        "name": "Hospital Gamma",
        "email": "admin@hospital-gamma.test",
        "password": "TestPassword123!"
    }
]

# Guardaremos los tokens aqui
tokens = {}

In [ ]:
# Registrar companies (si el endpoint existe)
# Si no, crearlos via Django admin o shell

for company in COMPANIES:
    resp = requests.post(
        f"{API_URL}/companies/register/",
        json={
            "name": company["name"],
            "email": company["email"],
            "password": company["password"],
            "password_confirm": company["password"]
        }
    )
    print_response(resp, f"Register {company['name']}")

### Si el registro no funciona, crear usuarios via Django shell:

```bash
docker compose exec django-dev python manage.py shell << 'EOF'
from apps.core.models import User, Company

companies_data = [
    {"name": "Hospital Alpha", "email": "admin@hospital-alpha.test"},
    {"name": "Hospital Beta", "email": "admin@hospital-beta.test"},
    {"name": "Hospital Gamma", "email": "admin@hospital-gamma.test"},
]

for data in companies_data:
    company, _ = Company.objects.get_or_create(
        email=data["email"],
        defaults={"name": data["name"]}
    )
    user, created = User.objects.get_or_create(
        email=data["email"],
        defaults={"company": company, "is_active": True}
    )
    if created:
        user.set_password("TestPassword123!")
        user.save()
    print(f"Created: {data['name']}")
EOF
```

## 3. Obtener tokens JWT

In [ ]:
def get_token(email, password):
    """Obtiene token JWT para un usuario."""
    resp = requests.post(
        f"{API_URL}/auth/token/",
        json={"email": email, "password": password}
    )
    if resp.status_code == 200:
        return resp.json()["access"]
    print(f"Error getting token for {email}: {resp.text}")
    return None

def auth_headers(token):
    """Headers con autenticacion."""
    return {"Authorization": f"Bearer {token}"}

# Obtener tokens para todos
for company in COMPANIES:
    token = get_token(company["email"], company["password"])
    if token:
        tokens[company["email"]] = token
        print(f"Token obtenido para {company['name']}")
    else:
        print(f"[ERROR] No se pudo obtener token para {company['name']}")

print(f"\nTokens obtenidos: {len(tokens)}/3")

## 4. Crear Consorcio (Hospital Alpha)

In [ ]:
# Hospital Alpha crea el consorcio
alpha_token = tokens.get("admin@hospital-alpha.test")

consortium_data = {
    "name": "Healthcare Research Consortium",
    "description": "Collaborative research on patient outcomes using privacy-preserving ML",
    "model_type": "logistic_regression",
    "min_members": 2,
    "voting_threshold": 0.6,
    "voting_duration_days": 7,
    "ml_config": {
        "learning_rate": 0.01,
        "epochs": 100,
        "batch_size": 32,
        "security_level": 128
    }
}

resp = requests.post(
    f"{API_URL}/consortiums/",
    json=consortium_data,
    headers=auth_headers(alpha_token)
)
print_response(resp, "Crear Consorcio")

if resp.status_code == 201:
    consortium = resp.json()
    CONSORTIUM_ID = consortium["id"]
    print(f"\nConsorcio creado con ID: {CONSORTIUM_ID}")
else:
    CONSORTIUM_ID = None

## 5. Invitar miembros al consorcio

In [ ]:
# Invitar a Hospital Beta y Gamma
invitations = []

for company in COMPANIES[1:]:  # Beta y Gamma
    invitation_data = {
        "consortium": CONSORTIUM_ID,
        "invitee_email": company["email"],
        "role": "member",
        "message": f"Invitacion para unirse al consorcio de investigacion",
        "expires_at": (datetime.now() + timedelta(days=30)).isoformat()
    }
    
    resp = requests.post(
        f"{API_URL}/consortiums/invitations/",
        json=invitation_data,
        headers=auth_headers(alpha_token)
    )
    print_response(resp, f"Invitar a {company['name']}")
    
    if resp.status_code == 201:
        invitations.append(resp.json())

print(f"\nInvitaciones enviadas: {len(invitations)}")

## 6. Aceptar invitaciones (Beta y Gamma)

In [ ]:
# Hospital Beta acepta la invitacion
beta_token = tokens.get("admin@hospital-beta.test")

# Primero, ver las invitaciones recibidas
resp = requests.get(
    f"{API_URL}/consortiums/invitations/received/",
    headers=auth_headers(beta_token)
)
print_response(resp, "Invitaciones recibidas (Beta)")

# Aceptar la primera invitacion
if resp.status_code == 200 and resp.json():
    invitation_id = resp.json()[0]["id"]
    resp = requests.post(
        f"{API_URL}/consortiums/invitations/{invitation_id}/accept/",
        headers=auth_headers(beta_token)
    )
    print_response(resp, "Beta acepta invitacion")

In [ ]:
# Hospital Gamma acepta la invitacion
gamma_token = tokens.get("admin@hospital-gamma.test")

resp = requests.get(
    f"{API_URL}/consortiums/invitations/received/",
    headers=auth_headers(gamma_token)
)

if resp.status_code == 200 and resp.json():
    invitation_id = resp.json()[0]["id"]
    resp = requests.post(
        f"{API_URL}/consortiums/invitations/{invitation_id}/accept/",
        headers=auth_headers(gamma_token)
    )
    print_response(resp, "Gamma acepta invitacion")

## 7. Verificar miembros del consorcio

In [ ]:
# Ver miembros del consorcio
resp = requests.get(
    f"{API_URL}/consortiums/{CONSORTIUM_ID}/members/",
    headers=auth_headers(alpha_token)
)
print_response(resp, "Miembros del Consorcio")

## 8. Registrar contribuciones de datos

In [ ]:
import hashlib

def generate_data_hash(data_description):
    """Genera un hash SHA-256 simulando datos."""
    return hashlib.sha256(data_description.encode()).hexdigest()

# Cada hospital contribuye datos
contributions = [
    {"token_key": "admin@hospital-alpha.test", "name": "Alpha", "records": 10000, "features": 50},
    {"token_key": "admin@hospital-beta.test", "name": "Beta", "records": 8000, "features": 50},
    {"token_key": "admin@hospital-gamma.test", "name": "Gamma", "records": 12000, "features": 50},
]

for contrib in contributions:
    token = tokens.get(contrib["token_key"])
    data_hash = generate_data_hash(f"{contrib['name']}-patient-data-2024")
    
    contribution_data = {
        "consortium": CONSORTIUM_ID,
        "record_count": contrib["records"],
        "feature_count": contrib["features"],
        "data_hash": data_hash,
        "checksum": generate_data_hash(f"checksum-{data_hash}")
    }
    
    resp = requests.post(
        f"{API_URL}/consortiums/{CONSORTIUM_ID}/contributions/",
        json=contribution_data,
        headers=auth_headers(token)
    )
    print_response(resp, f"Contribucion de {contrib['name']}")

## 9. Ver resumen de contribuciones

In [ ]:
# Resumen de contribuciones
resp = requests.get(
    f"{API_URL}/consortiums/{CONSORTIUM_ID}/contributions/summary/",
    headers=auth_headers(alpha_token)
)
print_response(resp, "Resumen de Contribuciones")

## 10. Ver estadisticas del consorcio

In [ ]:
# Estadisticas del consorcio
resp = requests.get(
    f"{API_URL}/consortiums/{CONSORTIUM_ID}/stats/",
    headers=auth_headers(alpha_token)
)
print_response(resp, "Estadisticas del Consorcio")

## 11. Iniciar entrenamiento del modelo

In [ ]:
# Iniciar entrenamiento (solo el owner/admin puede hacerlo)
resp = requests.post(
    f"{API_URL}/consortiums/{CONSORTIUM_ID}/start_training/",
    headers=auth_headers(alpha_token)
)
print_response(resp, "Iniciar Entrenamiento")

## 12. Verificar estado final

In [ ]:
# Ver el consorcio actualizado
resp = requests.get(
    f"{API_URL}/consortiums/{CONSORTIUM_ID}/",
    headers=auth_headers(alpha_token)
)
print_response(resp, "Estado Final del Consorcio")

## Resumen

Este notebook probó el flujo completo:

1. **Creacion de consorcio** - Hospital Alpha crea "Healthcare Research Consortium"
2. **Invitaciones** - Se invita a Hospital Beta y Gamma
3. **Aceptacion** - Ambos hospitales aceptan la invitacion
4. **Contribuciones** - Cada hospital registra sus datos (30,000 registros totales)
5. **Entrenamiento** - Se inicia el entrenamiento del modelo

### Endpoints utilizados:

| Endpoint | Metodo | Descripcion |
|----------|--------|-------------|
| `/api/v2/auth/token/` | POST | Obtener JWT |
| `/api/v2/consortiums/` | POST | Crear consorcio |
| `/api/v2/consortiums/invitations/` | POST | Enviar invitacion |
| `/api/v2/consortiums/invitations/received/` | GET | Ver invitaciones recibidas |
| `/api/v2/consortiums/invitations/{id}/accept/` | POST | Aceptar invitacion |
| `/api/v2/consortiums/{id}/members/` | GET | Ver miembros |
| `/api/v2/consortiums/{id}/contributions/` | POST | Registrar contribucion |
| `/api/v2/consortiums/{id}/contributions/summary/` | GET | Resumen de contribuciones |
| `/api/v2/consortiums/{id}/stats/` | GET | Estadisticas |
| `/api/v2/consortiums/{id}/start_training/` | POST | Iniciar entrenamiento |